### HW3 DSML

Name: Jeevika Mekala

Github Username: Jeevikaa

USC ID: 4369563404

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import re
import numpy as np
import pandas as pd
from scipy.stats import bootstrap

import warnings
warnings.filterwarnings('ignore')

### (a) Get the AReM Data Set

Data is in the AReM folder, one folder per activity. Files with formatting problems (like dataset4.csv in bending2) are handled by the reader below, which skips the # lines and splits on commas or spaces.

There are 88 instances and each one is a recording. A row is a time step and a column is one of the 6 time series (1: avg_rss12, 2: var_rss12, 3: avg_rss13, 4: var_rss13, 5: avg_rss23, 6: var_rss23). The number at the end of a feature name is the series number.

### (b) Test and Train Data

Datasets 1 and 2 of bending1 and bending2, and datasets 1, 2 and 3 of the other folders, are used as test data. Everything else is train data.

In [2]:
FILEPATH = r"C:\Users\jeevi\Downloads\USC\SEM-2\DSML\HW3\Homework 3, 4 Data\AReM"
COLS = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']


def read_file(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line == '' or line.startswith('#'):
                continue
            parts = [p for p in re.split(r'[,\s]+', line) if p != '']
            if len(parts) >= 7:
                rows.append([float(p) for p in parts[:7]])
    # first column is time, so it is dropped
    return pd.DataFrame(rows, columns=['time'] + COLS).drop(columns='time')


def file_number(name):
    return int(re.findall(r'\d+', name)[0])


def load_data(file_path):
    instances = []
    for folder in sorted(os.listdir(file_path)):
        path = os.path.join(file_path, folder)
        if not os.path.isdir(path):
            continue
        if folder in ('bending1', 'bending2'):
            test_ids = [1, 2]
        else:
            test_ids = [1, 2, 3]
        files = [f for f in os.listdir(path) if f.endswith('.csv')]
        for file in sorted(files, key=file_number):
            n = file_number(file)
            split = 'test' if n in test_ids else 'train'
            instances.append({'activity': folder, 'dataset': n, 'split': split,
                              'data': read_file(os.path.join(path, file))})
    return instances


instances = load_data(FILEPATH)
print("Total instances:", len(instances))
print("Train instances:", sum(i['split'] == 'train' for i in instances))
print("Test instances:", sum(i['split'] == 'test' for i in instances))

Total instances: 88
Train instances: 69
Test instances: 19


The table below shows the first 5 rows of the train data (all train recordings stacked together). Each row is one time step and each column is one of the 6 time series.

In [3]:
df_train = pd.concat([i['data'] for i in instances if i['split'] == 'train'], ignore_index=True)
df_test = pd.concat([i['data'] for i in instances if i['split'] == 'test'], ignore_index=True)
print(df_train.shape, df_test.shape)
df_train.head()

(33119, 6) (9120, 6)


,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,42.00,0.71,21.25,0.43,30.00,0.00
1,41.50,0.50,20.25,1.48,31.25,1.09
2,41.50,0.50,14.25,1.92,33.00,0.00
3,40.75,0.83,15.75,0.43,33.00,0.00
4,40.00,0.71,20.00,2.74,32.75,0.43


### (c) Feature Extraction

#### i. Research

1. Maximum - largest value in the series.
2. Minimum - smallest value in the series.
3. Mean - average of the values.
4. Median - the 50th percentile, not affected much by outliers.
5. Standard deviation - how spread out the values are around the mean.
6. First quartile - the 25th percentile.
7. Third quartile - the 75th percentile.
8. Interquartile range - third quartile minus first quartile.
9. Range - max minus min.
10. Cross-correlation - similarity between two series at different lags.
11. Auto-correlation - correlation of a series with a lagged copy of itself.
12. Skewness - how asymmetric the distribution of the values is.
13. Kurtosis - how heavy the tails of the distribution are.
14. Root mean square / energy - overall size of the signal.
15. Mean crossing rate - how often the series crosses its mean.
16. Orders of the AR, I and MA parts of a fitted ARIMA model.

#### ii. Extraction

Rows are the 88 instances and columns are the 42 features (7 statistics x 6 series). The number in the name is the series, so mean1 is the mean of avg_rss12.

In [4]:
stats = ['min', 'max', 'mean', 'median', 'standard deviation', '1st quartile', '3rd quartile']
feature_names = [s + str(i) for i in range(1, 7) for s in stats]


def get_features(d):
    row = []
    for col in COLS:
        x = d[col]
        row += [x.min(), x.max(), x.mean(), x.median(), x.std(), x.quantile(0.25), x.quantile(0.75)]
    return row


df = pd.DataFrame([get_features(i['data']) for i in instances], columns=feature_names)
df.index = df.index + 1
df.index.name = 'Instance'

# activity and train/test label for each instance, needed later for classification
labels = pd.DataFrame({'activity': [i['activity'] for i in instances],
                       'split': [i['split'] for i in instances]}, index=df.index)
df

,min1,max1,mean1,median1,standard deviation1,1st quartile1,3rd quartile1,min2,max2,mean2,...,standard deviation5,1st quartile5,3rd quartile5,min6,max6,mean6,median6,standard deviation6,1st quartile6,3rd quartile6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


#### iii. Standard Deviation

Each row below is one of the 42 features and the std column is the standard deviation of that feature across the 88 instances.

In [5]:
df.describe().loc[['std']].T

,std
min1,9.569975
max1,4.394362
mean1,5.335718
median1,5.440054
standard deviation1,1.772153
1st quartile1,6.153590
3rd quartile1,5.138925
min2,0.000000
max2,5.062729
mean2,1.574164


In the next table each row is again one feature. The columns are the standard deviation of the feature across the 88 instances, and the lower and upper end of its 90% bootstrap confidence interval.

In [6]:
def sample_std(x, axis=-1):
    return np.std(x, axis=axis, ddof=1)


rows = []
for feat in df.columns:
    res = bootstrap((df[feat].values,), sample_std, confidence_level=0.9, random_state=1, method='percentile')
    rows.append({'Feature': feat,
                 'Std': sample_std(df[feat].values),
                 'Lower Bound': res.confidence_interval.low,
                 'Upper Bound': res.confidence_interval.high})
confidence_intervals = pd.DataFrame(rows)
confidence_intervals

,Feature,Std,Lower Bound,Upper Bound
0,min1,9.569975,8.274250,10.748597
1,max1,4.394362,3.321675,5.294467
2,mean1,5.335718,4.702062,5.875852
3,median1,5.440054,4.788703,6.000721
4,standard deviation1,1.772153,1.573888,1.952432
5,1st quartile1,6.153590,5.573531,6.658507
6,3rd quartile1,5.138925,4.323253,5.828947
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.630393,5.409528
9,mean2,1.574164,1.397808,1.705502


#### iv. Select Features

The three features I chose are:

1. Mean: gives the average signal level of a series. The distance between the sensors is different for different activities, so the mean changes a lot between instances.
2. Maximum: the peak signal strength shows how large the movement was in an instance.
3. Standard deviation: shows how much the signal changes inside an instance. Activities like walking and cycling change a lot more than lying or sitting.

I did not choose the minimum because for the variance series (2, 4 and 6) it is almost always 0, so those features are nearly constant and their confidence interval is [0, 0]. The median is very close to the mean in this data so it does not add much.

## 2. ISLR 3.7.4

We fit a linear regression and a cubic regression to the same data.

### (a) Linear Train

The training RSS of the cubic regression will be lower than (or equal to) that of the linear regression. The linear model is a special case of the cubic model (set the x^2 and x^3 coefficients to 0). The cubic model is more flexible so it can always fit the training data at least as well, even if the true relationship is linear, because it will also fit some of the noise.

### (b) Linear Test

The test RSS of the linear regression will be lower than that of the cubic regression. The true relationship is linear, so the extra terms in the cubic model only fit noise in the training data (overfitting). This gives higher variance without any reduction in bias, so it does worse on new data.

### (c) Not Linear Train

The training RSS of the cubic regression will be lower than (or equal to) that of the linear regression. Same reason as in (a): the cubic model is more flexible, so it fits the training data at least as well as the linear model, no matter what the true relationship is.

### (d) Not Linear Testing

There is not enough information to tell. It depends on how far the true relationship is from linear. If it is close to linear, the linear model has lower variance and gives the lower test RSS. If it is far from linear, the cubic model has much lower bias and gives the lower test RSS.

### References

- James, Witten, Hastie, Tibshirani. An Introduction to Statistical Learning, Chapter 3, Exercise 4.
- https://stats.stackexchange.com/questions/50807/features-for-time-series-classification
- AReM data set, UCI Machine Learning Repository